# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya: Demonstrating AI-Ready Data Standards for Data Infrastructure in Africa Exploration with `mlcroissant`
This notebook demonstrates how to load, explore, and analyze the FAIR² Mental Health Survey Dataset using the [`mlcroissant`](https://pypi.org/project/mlcroissant/) library.

### Dataset Source
The dataset source is provided via a [Croissant schema](https://www.mlcommons.org/croissant/) URL.

In [ ]:
# Ensure the mlcroissant library is installed
!pip install -U mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the Croissant schema JSON-LD URL
croissant_url = "https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json"

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)

# Access the metadata as an object
metadata_json = dataset.metadata.to_json()

print(f"{metadata_json['name']}: {metadata_json['description']}")

## 2. Data Overview
Explore available record sets, fields and their IDs.
We enumerate all RecordSets, then their Fields/Columns by their `@id`, `name`, and other information.

In [ ]:
# List all record sets with their @id and name
print("Available RecordSets:")
record_sets = dataset.record_sets
for rs in record_sets:
    print(f"- @id: {rs.id}, name: {getattr(rs, 'name', 'N/A')}")

if not record_sets:
    print('No explicit RecordSets found in the metadata. Attempting to infer from files or top-level records...')

# Show fields/columns for first RecordSet, or list all fields from all RecordSets
for rs in record_sets:
    print(f"\nFields/Columns for RecordSet '@id': {rs.id}")
    for f in getattr(rs, 'fields', []):
        dtype = getattr(f, 'data_type', 'N/A')
        print(f"  - @id: {f.id}, name: {getattr(f, 'name', 'N/A')}, data_type: {dtype}")
    for c in getattr(rs, 'columns', []):
        dtype = getattr(c, 'data_type', 'N/A')
        print(f"  - @id: {c.id}, name: {getattr(c, 'name', 'N/A')}, data_type: {dtype}")

## 3. Data Extraction
Load data from record sets into DataFrames for analysis. Use the record set and field `@id`s from the overview above.

In [ ]:
# If there are no explicit RecordSets, mlcroissant's main record set is probably 'dataset',
# or you can load all records from top-level.

# Collect all present RecordSet @ids (using the id property for reference)
record_set_ids = [rs.id for rs in dataset.record_sets]

# If no record sets are present, try loading from the first data file if available
if not record_set_ids:
    print("No RecordSets found. Attempting to extract from main records ...")
    # mlcroissant allows loading records without specifying record_set
    record_set_ids = [None]  # Use None for the default

dataframes = {}

for record_set_id in record_set_ids:
    if record_set_id is not None:
        records = list(dataset.records(record_set=record_set_id))
    else:
        records = list(dataset.records()) # default
    print(f"Loaded {len(records)} records for record_set: {record_set_id}")
    # Only add non-empty data
    if records:
        dataframes[record_set_id or 'default'] = pd.DataFrame(records)

# Pick first DataFrame for demo
if dataframes:
    default_rs = list(dataframes.keys())[0]
    print("Columns in DataFrame:")
    print(dataframes[default_rs].columns.tolist())
    dataframes[default_rs].head()
else:
    print('No tabular data found in dataset!')

## 4. Exploratory Data Analysis (EDA)
Apply data processing steps such as filtering, normalizing numeric fields, and grouping. All field/column references use their schema `@id`.

We'll choose GAD-7 or PHQ-9 related fields as an example, assuming those are present (find the field via the column overview above!).

In [ ]:
# --- EDIT THIS BLOCK to match field @ids from the overview above ---

df = dataframes[default_rs]
# Try to pick a numeric assessment field, e.g., 'gad7_total_score' or similar.
# Find the exact column name from the column listing above (often the original CSV header or schema '@id').
possible_score_fields = [c for c in df.columns if "gad" in c.lower() or "phq" in c.lower() or "score" in c.lower()]
print("Detected possible score fields in DataFrame:", possible_score_fields)

# Select first such field
if possible_score_fields:
    numeric_field = possible_score_fields[0]
else:
    # Choose fallback: first float/integer column
    numeric_field = df.select_dtypes(include=["number"]).columns[0]

threshold = 10
filtered_df = df[df[numeric_field] > threshold]
print(f"Filtered records with {numeric_field} > {threshold}:")
print(filtered_df.head())

filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
print(f"Normalized {numeric_field} for filtered records:")
print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

# Grouping by a categorical field—find a likely option (e.g., 'gender' or 'village' or 'level_of_education')
possible_group_fields = [c for c in df.columns if 'gender' in c.lower() or 'village' in c.lower() or 'education' in c.lower()]
if possible_group_fields:
    group_field = possible_group_fields[0]
    grouped_df = filtered_df.groupby(group_field).mean(numeric_only=True)
    print(f"Grouped data by {group_field}:")
    print(grouped_df.head())
else:
    print('No obvious group field found in columns!')

## 5. Visualization
Visualize distributions and relationships between key survey assessment scores and demographics.

Below, we demonstrate basic plotting using Matplotlib. You can extend this with Seaborn or Altair.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(8, 5))
sns.histplot(df[numeric_field], bins=20, kde=True)
plt.title(f"Distribution of {numeric_field}")
plt.xlabel(numeric_field)
plt.ylabel('Count')
plt.show()

# Boxplot by a demographic variable if available
if possible_group_fields:
    group_var = possible_group_fields[0]
    plt.figure(figsize=(10, 4))
    sns.boxplot(data=df, x=group_var, y=numeric_field)
    plt.title(f"{numeric_field} by {group_var}")
    plt.ylabel(numeric_field)
    plt.xlabel(group_var)
    plt.show()

## 6. Conclusion
- This notebook demonstrated how to access, explore, and process the FAIR² Mental Health Survey Dataset from Kilifi County, Kenya using the `mlcroissant` library.
- We cataloged available record sets and fields using their `@id`s for future reproducibility.
- Sample EDA and visualizations illustrated how to analyze key mental health indicators.
- To extend, use specific field and record set `@id` values for deeper analyses, and cite the provided dataset appropriately for your research.